# Phase 4 — Yield engine (EIA-calibrated P50 CF)

**Model tournament:** Train multiple yield candidates (XGB, regularized XGB, Ridge, Huber, plant-year, physics) with spatial-block or plant-level CV, then **select the best OOF R²** per technology for cell scoring.

| Input | Output |
|-------|--------|
| `eia_facility_features.parquet` (`03`) | `cell_p50_yield.parquet` |
| `cell_physics_yield.parquet` (`01b`) | `models/yield_solar_p50.joblib`, `yield_wind_p50.joblib` |
| `cells_passing_gateway.parquet` (`06`) | `yield_*_cv_report.json`, `maps/*_yield_p50_ny.html` |

**Requires:** `01`, `01b`, `03`, `06`.

In [1]:
N_CV_FOLDS = 5
RANDOM_STATE = 42
SPATIAL_BLOCK_DEG = 0.5
TRAIN_YEAR_MIN, TRAIN_YEAR_MAX = 2018, 2023
TARGET_CRS = 'EPSG:32618'

import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
import xgboost as xgb
from shapely.geometry import Point
from copy import deepcopy
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path('../..').resolve()
EIA_FEATURES_PATH = ROOT / 'data' / 'processed' / 'eia_facility_features.parquet'
PHYSICS_PATH = ROOT / 'data' / 'processed' / 'cell_physics_yield.parquet'
GATEWAY_PATH = ROOT / 'data' / 'processed' / 'cells_passing_gateway.parquet'
OUT_PATH = ROOT / 'data' / 'processed' / 'cell_p50_yield.parquet'
NY_BOUNDARY_PATH = ROOT / 'data' / 'ref' / 'ny_state_boundary.geojson'
MAP_DIR = ROOT / 'maps'

XGB_PARAMS = dict(
    n_estimators=120, max_depth=4, learning_rate=0.08,
    subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE,
    objective='reg:squarederror', tree_method='hist', n_jobs=4,
)
XGB_PARAMS_REG = dict(
    n_estimators=80, max_depth=2, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=5.0,
    random_state=RANDOM_STATE, objective='reg:squarederror', tree_method='hist', n_jobs=4,
)

for p in (EIA_FEATURES_PATH, GATEWAY_PATH, PHYSICS_PATH):
    if not p.exists():
        raise FileNotFoundError(p)
(ROOT / 'models').mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)

ROOT: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York


In [ ]:
MIN_NAMEPLATE_MW = 1.0  # default print filter
MIN_NAMEPLATE_MW_BY_TECH = {'solar': 5.0, 'wind': 1.0}
SCORING_SOLAR_NAMEPLATE_MW = 14.0  # median utility-scale MW for grid-cell scoring

eia_features = pl.read_parquet(EIA_FEATURES_PATH)
train_base = eia_features.filter(
    pl.col('cf_valid_flag')
    & pl.col('year').is_between(TRAIN_YEAR_MIN, TRAIN_YEAR_MAX)
    & (pl.col('nameplate_mw') >= MIN_NAMEPLATE_MW)
)
print('Plant-year training rows (>=', MIN_NAMEPLATE_MW, 'MW):', train_base.height)
print(train_base.group_by('technology').agg(
    pl.len().alias('rows'),
    pl.col('plant_id').n_unique().alias('plants'),
    pl.col('capacity_factor').mean().alias('mean_cf'),
))

Plant-year training rows (>= 1.0 MW): 1644
shape: (2, 4)
┌────────────┬──────┬────────┬──────────┐
│ technology ┆ rows ┆ plants ┆ mean_cf  │
│ ---        ┆ ---  ┆ ---    ┆ ---      │
│ str        ┆ u32  ┆ u32    ┆ f64      │
╞════════════╪══════╪════════╪══════════╡
│ solar      ┆ 1490 ┆ 432    ┆ 0.026722 │
│ wind       ┆ 154  ┆ 31     ┆ 0.023749 │
└────────────┴──────┴────────┴──────────┘


In [3]:
def spatial_block(lat, lon):
    lat = pd.to_numeric(lat, errors='coerce').fillna(0)
    lon = pd.to_numeric(lon, errors='coerce').fillna(0)
    return (np.floor(lat / SPATIAL_BLOCK_DEG) * 10_000 + np.floor(lon / SPATIAL_BLOCK_DEG)).astype(int)


def standardize_latlon(pdf):
    pdf = pdf.loc[:, ~pdf.columns.duplicated(keep='first')].copy()
    if 'Latitude' not in pdf.columns and 'latitude' in pdf.columns:
        pdf['Latitude'] = pdf['latitude']
    if 'Longitude' not in pdf.columns and 'longitude' in pdf.columns:
        pdf['Longitude'] = pdf['longitude']
    return pdf


def plant_year_train(technology, features, min_mw=None):
    min_mw = MIN_NAMEPLATE_MW_BY_TECH.get(technology, 1.0) if min_mw is None else min_mw
    sub = train_base.filter(
        (pl.col('technology') == technology)
        & (pl.col('nameplate_mw') >= min_mw)
    )
    pdf = standardize_latlon(sub.to_pandas())
    keep = list(dict.fromkeys(['plant_id', 'year', 'capacity_factor', 'nameplate_mw', 'grid_latitude', 'grid_longitude', *features]))
    keep = [c for c in keep if c in pdf.columns]
    return pdf[keep].dropna(subset=['Latitude', 'Longitude'] + features + ['capacity_factor', 'plant_id'])


def plant_level_train(technology, features, min_mw=None):
    min_mw = MIN_NAMEPLATE_MW_BY_TECH.get(technology, 1.0) if min_mw is None else min_mw
    sub = eia_features.filter(
        pl.col('cf_valid_flag')
        & (pl.col('technology') == technology)
        & (pl.col('nameplate_mw') >= min_mw)
    )
    pdf = standardize_latlon(sub.to_pandas())
    keep = list(dict.fromkeys(['plant_id', 'capacity_factor', 'nameplate_mw', 'grid_latitude', 'grid_longitude', *features]))
    keep = [c for c in keep if c in pdf.columns]
    pdf = pdf[keep].dropna(subset=['Latitude', 'Longitude'] + features + ['capacity_factor', 'plant_id'])
    agg = {c: 'first' for c in features + ['nameplate_mw', 'grid_latitude', 'grid_longitude']}
    agg['capacity_factor'] = 'mean'
    grouped = pdf.groupby('plant_id', as_index=False).agg(agg)
    return grouped.dropna(subset=['Latitude', 'Longitude'] + features + ['capacity_factor'])


def attach_physics(pdf, tech):
    phy = pl.read_parquet(PHYSICS_PATH).to_pandas()
    key = 'physics_solar_cf_p50' if tech == 'solar' else 'physics_wind_cf_p50'
    phy = phy.rename(columns={'latitude': 'grid_latitude', 'longitude': 'grid_longitude', key: 'physics_cf'})
    return pdf.merge(phy[['grid_latitude', 'grid_longitude', 'physics_cf']], on=['grid_latitude', 'grid_longitude'], how='left')


def cv_groups(df, cv_group):
    if cv_group == 'plant_id':
        return df['plant_id'].values
    return spatial_block(df['Latitude'], df['Longitude'])


def run_feature_cv(df, features, model_factory, cv_group):
    X = df[features].astype(np.float32)
    y = df['capacity_factor'].astype(np.float32)
    groups = cv_groups(df, cv_group)
    n_splits = min(N_CV_FOLDS, len(np.unique(groups)))
    if n_splits < 2:
        return None, {'r2': float('nan'), 'mae': float('nan'), 'rmse': float('nan')}, []
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(df))
    folds = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups)):
        m = model_factory()
        m.fit(X.iloc[tr], y.iloc[tr])
        pred = np.clip(m.predict(X.iloc[te]), 0, 1)
        oof[te] = pred
        folds.append({'fold': fold, 'r2': float(r2_score(y.iloc[te], pred)), 'mae': float(mean_absolute_error(y.iloc[te], pred))})
    cv = {
        'r2': float(r2_score(y, oof)),
        'mae': float(mean_absolute_error(y, oof)),
        'rmse': float(np.sqrt(mean_squared_error(y, oof))),
    }
    return oof, cv, folds


def run_physics_raw_cv(df, tech):
    d = attach_physics(df, tech).dropna(subset=['physics_cf'])
    if len(d) < 10:
        return {'r2': float('nan'), 'mae': float('nan'), 'rmse': float('nan')}, []
    y = d['capacity_factor'].astype(np.float32)
    pred = np.clip(d['physics_cf'].astype(np.float32), 0, 1)
    return {
        'r2': float(r2_score(y, pred)),
        'mae': float(mean_absolute_error(y, pred)),
        'rmse': float(np.sqrt(mean_squared_error(y, pred))),
    }, []


def run_physics_cal_cv(df, tech, cv_group):
    d = attach_physics(df, tech).dropna(subset=['physics_cf'])
    if len(d) < 10:
        return {'r2': float('nan'), 'mae': float('nan'), 'rmse': float('nan')}, []
    X = d[['physics_cf']].astype(np.float32)
    y = d['capacity_factor'].astype(np.float32)
    groups = cv_groups(d, cv_group)
    n_splits = min(N_CV_FOLDS, len(np.unique(groups)))
    if n_splits < 2:
        return {'r2': float('nan'), 'mae': float('nan'), 'rmse': float('nan')}, []
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(d))
    folds = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups)):
        m = Ridge(alpha=1.0)
        m.fit(X.iloc[tr], y.iloc[tr])
        pred = np.clip(m.predict(X.iloc[te]), 0, 1)
        oof[te] = pred
        folds.append({'fold': fold, 'r2': float(r2_score(y.iloc[te], pred)), 'mae': float(mean_absolute_error(y.iloc[te], pred))})
    return {
        'r2': float(r2_score(y, oof)),
        'mae': float(mean_absolute_error(y, oof)),
        'rmse': float(np.sqrt(mean_squared_error(y, oof))),
    }, folds


SOLAR_FEATURES_V1 = [
    'Latitude', 'Longitude', 'shortwave_mean', 'direct_mean', 'diffuse_mean',
    'temperature_mean', 'surface_pressure_mean',
]
WIND_FEATURES_V1 = [
    'Latitude', 'Longitude', 'wind_power_mean', 'wind_speed_mean',
    'temperature_mean', 'surface_pressure_mean',
]
SOLAR_FEATURES_V2 = SOLAR_FEATURES_V1 + [
    'shortwave_diurnal_amplitude', 'clear_sky_index', 'direct_diffuse_ratio',
]
WIND_FEATURES_V2 = WIND_FEATURES_V1 + [
    'wind_speed_hub_mean', 'wind_hub_speed_p90', 'wind_ramp_p95', 'weibull_k',
]


def run_ensemble_cv(df, features, factories, weights, cv_group):
    X = df[features].astype(np.float32)
    y = df['capacity_factor'].astype(np.float32)
    groups = cv_groups(df, cv_group)
    n_splits = min(N_CV_FOLDS, len(np.unique(groups)))
    if n_splits < 2:
        return {'r2': float('nan'), 'mae': float('nan'), 'rmse': float('nan')}, []
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(df))
    folds = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups)):
        preds = []
        for factory in factories:
            m = factory()
            m.fit(X.iloc[tr], y.iloc[tr])
            preds.append(np.clip(m.predict(X.iloc[te]), 0, 1))
        pred = sum(w * p for w, p in zip(weights, preds))
        oof[te] = pred
        folds.append({'fold': fold, 'r2': float(r2_score(y.iloc[te], pred)), 'mae': float(mean_absolute_error(y.iloc[te], pred))})
    cv = {
        'r2': float(r2_score(y, oof)),
        'mae': float(mean_absolute_error(y, oof)),
        'rmse': float(np.sqrt(mean_squared_error(y, oof))),
    }
    return cv, folds


SOLAR_FEATURES_V2_MW = SOLAR_FEATURES_V2 + ['nameplate_mw']

ENSEMBLE_CONFIGS = {
    'ens_xgb_ridge_spatial': {
        'technologies': ['wind'],
        'training_unit': 'one_row_per_plant',
        'feature_set': 'v1',
        'cv_group': 'spatial_block',
        'weights': [0.6, 0.4],
        'factories': [
            lambda: xgb.XGBRegressor(**XGB_PARAMS),
            lambda: Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=10.0))]),
        ],
    },
    'ens_xgb_ridge_v2_spatial': {
        'technologies': ['wind'],
        'training_unit': 'one_row_per_plant',
        'feature_set': 'v2',
        'cv_group': 'spatial_block',
        'weights': [0.72, 0.28],
        'factories': [
            lambda: xgb.XGBRegressor(**XGB_PARAMS),
            lambda: Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=20.0))]),
        ],
    },
    'ens_ridge05_xgb_mw_plant': {
        'technologies': ['solar'],
        'training_unit': 'plant_year',
        'feature_set': 'v2mw',
        'cv_group': 'plant_id',
        'weights': [0.50, 0.50],
        'factories': [
            lambda: Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=0.5))]),
            lambda: xgb.XGBRegressor(**XGB_PARAMS_REG),
        ],
    },
}

MODEL_SPECS = [
  # name, training_unit, feature_set, cv_group, kind, factory
  ('xgb_plant_v1_spatial', 'one_row_per_plant', 'v1', 'spatial_block', 'feature',
   lambda: xgb.XGBRegressor(**XGB_PARAMS)),
  ('xgb_plant_v1_plant', 'one_row_per_plant', 'v1', 'plant_id', 'feature',
   lambda: xgb.XGBRegressor(**XGB_PARAMS)),
  ('xgb_reg_plant_v1_spatial', 'one_row_per_plant', 'v1', 'spatial_block', 'feature',
   lambda: xgb.XGBRegressor(**XGB_PARAMS_REG)),
  ('xgb_reg_plant_v1_plant', 'one_row_per_plant', 'v1', 'plant_id', 'feature',
   lambda: xgb.XGBRegressor(**XGB_PARAMS_REG)),
  ('ridge_plant_v1_spatial', 'one_row_per_plant', 'v1', 'spatial_block', 'feature',
   lambda: Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=10.0))])),
  ('ridge_plant_v1_plant', 'one_row_per_plant', 'v1', 'plant_id', 'feature',
   lambda: Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=10.0))])),
  ('huber_plant_v1_plant', 'one_row_per_plant', 'v1', 'plant_id', 'feature',
   lambda: Pipeline([('s', StandardScaler()), ('m', HuberRegressor())])),
  ('xgb_year_v2_spatial', 'plant_year', 'v2', 'spatial_block', 'feature',
   lambda: xgb.XGBRegressor(**XGB_PARAMS)),
  ('xgb_year_v2_plant', 'plant_year', 'v2', 'plant_id', 'feature',
   lambda: xgb.XGBRegressor(**XGB_PARAMS)),
  ('ridge_year_v2_plant', 'plant_year', 'v2', 'plant_id', 'feature',
   lambda: Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=10.0))])),
  ('ridge_year_v2_mw_plant', 'plant_year', 'v2mw', 'plant_id', 'feature',
   lambda: Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))])),
  ('ridge_year_v2_mw_plant_a05', 'plant_year', 'v2mw', 'plant_id', 'feature',
   lambda: Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=0.5))])),
  ('ens_ridge05_xgb_mw_plant', 'plant_year', 'v2mw', 'plant_id', 'ensemble', None),
  ('ens_xgb_ridge_spatial', 'one_row_per_plant', 'v1', 'spatial_block', 'ensemble', None),
  ('ens_xgb_ridge_v2_spatial', 'one_row_per_plant', 'v2', 'spatial_block', 'ensemble', None),
  ('physics_raw_plant', 'one_row_per_plant', 'physics', 'spatial_block', 'physics_raw', None),
  ('physics_cal_plant_spatial', 'one_row_per_plant', 'physics', 'spatial_block', 'physics_cal', None),
  ('physics_cal_plant', 'one_row_per_plant', 'physics', 'plant_id', 'physics_cal', None),
]

feature_sets = {
    'solar': {'v1': SOLAR_FEATURES_V1, 'v2': SOLAR_FEATURES_V2, 'v2mw': SOLAR_FEATURES_V2_MW},
    'wind': {'v1': WIND_FEATURES_V1, 'v2': WIND_FEATURES_V2},
}

models = {}
reports = {}

for tech in ('solar', 'wind'):
    train_plant = plant_level_train(tech, feature_sets[tech]['v1'])
    train_plant_v2 = plant_level_train(tech, feature_sets[tech]['v2'])
    train_year = plant_year_train(tech, feature_sets[tech]['v2'])
    train_year_mw = plant_year_train('solar', feature_sets['solar']['v2mw']) if tech == 'solar' else None
    print(f'{tech}: plant-level={train_plant.plant_id.nunique()} | plant-year={len(train_year)} ({train_year.plant_id.nunique()} plants)')

    def training_frame(unit, fset):
        if fset == 'v2mw':
            return train_year_mw
        if unit == 'one_row_per_plant':
            return train_plant_v2 if fset == 'v2' else train_plant
        return train_year

    leaderboard = []
    for name, unit, fset, cv_group, kind, factory in MODEL_SPECS:
        if fset == 'v2mw' and tech != 'solar':
            continue
        if kind == 'ensemble':
            cfg = ENSEMBLE_CONFIGS.get(name)
            if cfg is None or tech not in cfg['technologies']:
                continue
        if kind == 'physics_raw':
            df = train_plant
            cv, folds = run_physics_raw_cv(df, tech)
        elif kind == 'physics_cal':
            df = train_plant
            cv, folds = run_physics_cal_cv(df, tech, cv_group)
        elif kind == 'ensemble':
            cfg = ENSEMBLE_CONFIGS[name]
            df = training_frame(cfg['training_unit'], cfg['feature_set'])
            feats = feature_sets[tech][cfg['feature_set']]
            cv, folds = run_ensemble_cv(df, feats, cfg['factories'], cfg['weights'], cfg['cv_group'])
            unit, fset, cv_group = cfg['training_unit'], cfg['feature_set'], cfg['cv_group']
        else:
            df = training_frame(unit, fset)
            feats = feature_sets[tech][fset]
            _, cv, folds = run_feature_cv(df, feats, factory, cv_group)
        leaderboard.append({
            'model': name,
            'training_unit': unit,
            'feature_set': fset,
            'cv_group': cv_group,
            'kind': kind,
            'cv_oof': cv,
            'cv_folds': folds,
        })
        print(f'  {name:28s} R²={cv["r2"]:7.3f}  MAE={cv["mae"]:.4f}')

    winner = max(
        leaderboard,
        key=lambda row: (
            row['cv_oof']['r2'] if row['cv_oof']['r2'] == row['cv_oof']['r2'] else -999.0,
            -row['cv_oof']['mae'] if row['cv_oof']['mae'] == row['cv_oof']['mae'] else -999.0,
        ),
    )
    print(f'  --> SELECTED: {winner["model"]}  R²={winner["cv_oof"]["r2"]:.3f}')

    w = winner
    if w['kind'] == 'physics_raw':
        fit_df = attach_physics(train_plant, tech).dropna(subset=['physics_cf'])
        fitted = {'kind': 'physics_raw', 'tech': tech}
        train_df = fit_df
        score_features = []
    elif w['kind'] == 'physics_cal':
        fit_df = attach_physics(train_plant, tech).dropna(subset=['physics_cf'])
        fitted = Ridge(alpha=1.0)
        fitted.fit(fit_df[['physics_cf']].astype(np.float32), fit_df['capacity_factor'].astype(np.float32))
        train_df = fit_df
        score_features = ['physics_cf']
    elif w['kind'] == 'ensemble':
        cfg = ENSEMBLE_CONFIGS[w['model']]
        train_df = training_frame(cfg['training_unit'], cfg['feature_set'])
        score_features = feature_sets[tech][cfg['feature_set']]
        fitted = []
        X_fit = train_df[score_features].astype(np.float32)
        y_fit = train_df['capacity_factor'].astype(np.float32)
        for factory in cfg['factories']:
            m = factory()
            m.fit(X_fit, y_fit)
            fitted.append(m)
    else:
        train_df = training_frame(w['training_unit'], w['feature_set'])
        score_features = feature_sets[tech][w['feature_set']]
        spec = next(s for s in MODEL_SPECS if s[0] == w['model'])
        fitted = spec[5]()
        fitted.fit(train_df[score_features].astype(np.float32), train_df['capacity_factor'].astype(np.float32))

    models[tech] = {
        'model': fitted,
        'kind': w['kind'],
        'features': score_features if w['kind'] != 'physics_raw' else [],
        'training_unit': w['training_unit'],
        'selected_model': w['model'],
        'ensemble_weights': ENSEMBLE_CONFIGS[w['model']]['weights'] if w['kind'] == 'ensemble' else None,
        'scoring_nameplate_mw': SCORING_SOLAR_NAMEPLATE_MW if tech == 'solar' and 'nameplate_mw' in (score_features if w['kind'] not in ('physics_raw',) else []) else None,
    }

    n_plants = int(train_year.plant_id.nunique()) if w['training_unit'] == 'plant_year' else int(train_plant.plant_id.nunique())
    report = {
        'cv_group': w['cv_group'],
        'training_unit': w['training_unit'],
        'selected_model': w['model'],
        'model_kind': w['kind'],
        'cv_oof': w['cv_oof'],
        'cv_folds': w['cv_folds'],
        'model_leaderboard': [
            {'model': r['model'], 'cv_group': r['cv_group'], 'training_unit': r['training_unit'], **r['cv_oof']}
            for r in sorted(leaderboard, key=lambda r: r['cv_oof']['r2'], reverse=True)
        ],
        'technology': tech,
        'features': score_features if score_features else [f'physics_{tech}_cf_p50'],
        'n_plant_years': len(train_year),
        'n_plants': n_plants,
        'min_nameplate_mw': MIN_NAMEPLATE_MW_BY_TECH.get(tech, 1.0),
        'scoring_nameplate_mw': models[tech].get('scoring_nameplate_mw'),
        'selection_rule': 'max_oof_r2_then_min_mae',
    }
    reports[tech] = report
    joblib.dump(models[tech], ROOT / 'models' / f'yield_{tech}_p50.joblib')
    (ROOT / 'data' / 'processed' / f'yield_{tech}_cv_report.json').write_text(json.dumps(report, indent=2))

solar: plant-level=472 | plant-year=1463 (412 plants)


  xgb_plant_v1_spatial         R²= -0.283  MAE=0.0227


  xgb_plant_v1_plant           R²= -0.089  MAE=0.0192
  xgb_reg_plant_v1_spatial     R²= -0.004  MAE=0.0197


  xgb_reg_plant_v1_plant       R²= -0.007  MAE=0.0197
  ridge_plant_v1_spatial       R²= -0.039  MAE=0.0201
  ridge_plant_v1_plant         R²= -0.021  MAE=0.0198


  huber_plant_v1_plant         R²= -0.041  MAE=0.0187


...

  huber_plant_v1_plant         R²= -0.413  MAE=0.0311


  xgb_year_v2_spatial          R²= -1.164  MAE=0.0188


  xgb_year_v2_plant            R²= -2.260  MAE=0.0206
  ridge_year_v2_plant          R²= -0.194  MAE=0.0131


  ens_xgb_ridge_spatial        R²=  0.073  MAE=0.0283


  ens_xgb_ridge_v2_spatial     R²=  0.113  MAE=0.0274
  physics_raw_plant            R²= -2.413  MAE=0.0723
  physics_cal_plant_spatial    R²= -0.147  MAE=0.0353
  physics_cal_plant            R²= -0.043  MAE=0.0329
  --> SELECTED: ens_xgb_ridge_v2_spatial  R²=0.113


In [4]:
cells = pl.read_parquet(GATEWAY_PATH).filter(pl.col('in_ny_state')).to_pandas()
cells = cells.drop(columns=['index_right'], errors='ignore')
physics_cells = pl.read_parquet(PHYSICS_PATH).to_pandas()
out = cells[['latitude', 'longitude']].drop_duplicates().copy()

feat_map = {
    'Latitude': 'latitude', 'Longitude': 'longitude',
    'shortwave_mean': 'shortwave_mean', 'direct_mean': 'direct_mean', 'diffuse_mean': 'diffuse_mean',
    'shortwave_diurnal_amplitude': 'shortwave_diurnal_amplitude',
    'clear_sky_index': 'clear_sky_index', 'direct_diffuse_ratio': 'direct_diffuse_ratio',
    'wind_power_mean': 'wind_power_mean', 'wind_speed_mean': 'wind_speed_mean',
    'wind_speed_hub_mean': 'wind_speed_hub_mean', 'wind_hub_speed_p90': 'wind_hub_speed_p90',
    'wind_ramp_p95': 'wind_ramp_p95', 'weibull_k': 'weibull_k',
    'temperature_mean': 'temperature_mean', 'surface_pressure_mean': 'surface_pressure_mean',
}

for tech, col in [('solar', 'solar_cf_p50'), ('wind', 'wind_cf_p50')]:
    meta = models[tech]
    kind = meta['kind']
    if kind == 'physics_raw':
        phy_col = 'physics_solar_cf_p50' if tech == 'solar' else 'physics_wind_cf_p50'
        scored = physics_cells[['latitude', 'longitude', phy_col]].rename(columns={phy_col: col})
        out = out.merge(scored, on=['latitude', 'longitude'], how='left')
        out[col] = np.clip(out[col], 0, 1)
    elif kind == 'physics_cal':
        phy_col = 'physics_solar_cf_p50' if tech == 'solar' else 'physics_wind_cf_p50'
        scored = physics_cells[['latitude', 'longitude', phy_col]].copy()
        scored['physics_cf'] = scored[phy_col]
        pred = np.clip(meta['model'].predict(scored[['physics_cf']].astype(np.float32)), 0, 1)
        scored = scored[['latitude', 'longitude']].assign(**{col: pred})
        out = out.merge(scored, on=['latitude', 'longitude'], how='left')
    elif kind == 'ensemble':
        weather_feats = [f for f in meta['features'] if f != 'nameplate_mw']
        need = [feat_map[f] for f in weather_feats]
        sub = cells.dropna(subset=need).copy()
        X = pd.DataFrame({f: sub[feat_map[f]] for f in weather_feats})
        if meta.get('scoring_nameplate_mw') is not None:
            X['nameplate_mw'] = meta['scoring_nameplate_mw']
        X = X[meta['features']]
        preds = [np.clip(m.predict(X.astype(np.float32)), 0, 1) for m in meta['model']]
        pred = sum(w * p for w, p in zip(meta['ensemble_weights'], preds))
        scored = pd.DataFrame({'latitude': sub['latitude'], 'longitude': sub['longitude'], col: pred})
        out = out.merge(scored, on=['latitude', 'longitude'], how='left')
    else:
        weather_feats = [f for f in meta['features'] if f != 'nameplate_mw']
        need = [feat_map[f] for f in weather_feats]
        sub = cells.dropna(subset=need).copy()
        X = pd.DataFrame({f: sub[feat_map[f]] for f in weather_feats})
        if meta.get('scoring_nameplate_mw') is not None:
            X['nameplate_mw'] = meta['scoring_nameplate_mw']
        X = X[meta['features']]
        pred = np.clip(meta['model'].predict(X.astype(np.float32)), 0, 1)
        scored = pd.DataFrame({'latitude': sub['latitude'], 'longitude': sub['longitude'], col: pred})
        out = out.merge(scored, on=['latitude', 'longitude'], how='left')

    out[f'{col}_model'] = meta['selected_model']
    valid = out[col].notna().sum()
    print(f'{tech}: {meta["selected_model"]} scored {valid} cells, CF range {out[col].min():.4f}-{out[col].max():.4f}')

if 'solar_cf_p50' in out.columns and 'wind_cf_p50' in out.columns:
    out['hybrid_cf_p50'] = (out['solar_cf_p50'].fillna(0) + out['wind_cf_p50'].fillna(0)) / 2

pl.from_pandas(out).write_parquet(OUT_PATH)
print('Wrote', OUT_PATH, '| rows', len(out))
out.describe()

solar: ens_ridge05_xgb_mw_plant scored 1089 cells, CF range 0.0221-0.0343
wind: ens_xgb_ridge_v2_spatial scored 1089 cells, CF range 0.0132-0.1178
Wrote /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/cell_p50_yield.parquet | rows 1089


,latitude,longitude,solar_cf_p50,wind_cf_p50,hybrid_cf_p50
count,1089.000000,1089.000000,1089.000000,1089.000000,1089.000000
mean,42.821405,-75.947438,0.028397,0.053081,0.040739
std,0.793164,1.675755,0.002097,0.027319,0.013938
min,41.220000,-79.710000,0.022112,0.013185,0.019645
25%,42.300000,-77.280000,0.027121,0.030400,0.029035
50%,42.660000,-75.750000,0.028569,0.039193,0.033912
75%,43.200000,-74.580000,0.029808,0.082485,0.055901
max,44.910000,-73.320000,0.034345,0.117839,0.075008


In [5]:
ny = gpd.read_file(NY_BOUNDARY_PATH)
outline = ny.boundary.iloc[0]

for col, title, fname in [
    ('solar_cf_p50', 'Solar P50 yield (predicted CF)', 'solar_yield_p50_ny.html'),
    ('wind_cf_p50', 'Wind P50 yield (predicted CF)', 'wind_yield_p50_ny.html'),
]:
    if col not in out.columns:
        continue
    m = out.dropna(subset=[col])
    fig = go.Figure()
    fig.add_trace(go.Scattermap(
        lat=m['latitude'], lon=m['longitude'], mode='markers',
        marker=dict(size=8, color=m[col], colorscale='YlOrRd', showscale=True, opacity=0.7),
        text=[f'CF={v:.3f}' for v in m[col]],
    ))
    fig.add_trace(go.Scattermap(
        lat=list(outline.xy[1]), lon=list(outline.xy[0]),
        mode='lines', line=dict(width=2, color='#333'), hoverinfo='skip', showlegend=False,
    ))
    fig.update_layout(map=dict(style='carto-positron', center=dict(lat=42.9, lon=-75.8), zoom=5.5), title=title, height=800)
    path = MAP_DIR / fname
    fig.write_html(str(path), include_plotlyjs='cdn')
    print('Map ->', path)

Map -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/maps/solar_yield_p50_ny.html
Map -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/maps/wind_yield_p50_ny.html
